# 🏠 房价预测：基于 PyTorch 的单变量线性回归 (扩展篇)

湖北理工学院《人工智能理论》课程资料

📝 **本节实验包含了更多进阶与扩展内容：**
- 🚀 学习率的深入剖析（为什么会发散、震荡）
- 🛡️ 模型泛化能力验证
- 📉 小样本下的过拟合现象及缓解对策
- ⚖️ 正则化 (L1 / L2) 的影响
- 🔄 多种交叉验证方法 (Cross-Validation)

## 📊 1. 获取并浏览房价数据

首先，我们将读取本地整理好的波士顿房价数据 (`./data/houseprice.csv`)。

原始数据包含 13 种描述房屋和周边环境的特征 (Features) 和 1 个我们想要预测的目标房价 (Target)。

**全特征字典 (Features)：**
- `[0]`: 按城镇划分的犯罪率
- `[1]`: 大住宅用地占比
- `[2]`: 非零售商业用地占比
- `[3]`: 景观房 (是否位于查尔斯河畔)
- `[4]`: 氮氧化物浓度（空气质量） 
- `[5]`: **每个住宅的平均房间数 👉 (默认探索特征)**
- `[6]`: 老旧房屋占比
- `[7]`: 离就业中心的加权距离
- `[8]`: 辐射路可达性指数
- `[9]`: 每万元房产税
- `[10]`: 学生-教师比
- `[11]`: 黑人城镇比例相关的统计指标
- `[12]`: 低层次人口占比

**目标预测标量 (Target)：**
- **自住房房价中位数（单位：千美元）** 👉 **(我们的预测目标 y)**

下面的代码不仅会提取 `(x, y)` 供后续训练，还会顺便画出**所有 13 个特征与房价之间的散点图关系矩阵**，供你直观地探索整个数据集的全貌！

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

# ========================================================
# 👇 设置中文字体，确保图表正常显示汉字
# ========================================================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========================================================
# 👇 导入数据并整理全部特征
# ========================================================
data_url = './data/houseprice.csv'
raw_df = pd.read_csv(data_url, sep=',', skiprows=1, header=None)

data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

# 提取单一主特征构建模型：[5] 平均房间数
x = data[:, 5]
y = target

print(f"✅ 数据加载完成！样本总数: {len(x)}")
print(f"示例：房间数 = {x[0]:.2f}, 房价 = {y[0]:.2f}千美元\n")

# ========================================================
# 👇 绘制全部 13 个特征的散点图矩阵
# ========================================================
fts_names = [
    '犯罪率（%）', '大住宅用地占比（%）', '非零售商业用地占比（%）',
    '景观房', '氮氧化物浓度（ppm）', '平均房间数',
    '老旧房屋占比（%）', '离就业中心的加权距离', '辐射路可达性指标',
    '每万元房产税', '学生-教师比', '黑人比例统计',
    '低层次人口占比（%）'
]

num_fts = data.shape[1]
num_col = 5
num_row = int(np.ceil(num_fts / num_col))

fig, axes = plt.subplots(num_row, num_col, figsize=(20, 3.5 * num_row))
fig.suptitle("13个房屋特征与房价中位数的散点关系一览表", fontsize=20, fontweight='bold')

for i in range(num_fts):
    row, col = i // num_col, i % num_col
    ax = axes[row, col]
    ax.scatter(data[:, i], target, color='teal', alpha=0.5, edgecolors='k')
    ax.set_xlabel(fts_names[i], fontsize=13)
    ax.set_ylabel('房价中位数', fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.3)

# 隐藏多余的空白子图
for j in range(num_fts, num_row * num_col):
    fig.delaxes(axes[j // num_col, j % num_col])

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

## 💡 2. 学习率对训练结果的影响 (深入动态剖析)

在梯度下降中，**学习率**决定了参数更新的步长大小。
- 步长**太小**：模型训练极其缓慢，由于收敛过慢甚至可能半路停止。
- 步长**适中**：模型能够在合理的迭代次数内快速平稳地到达最优解。
- 步长**过大**：跨度太大，直接跨过最低点，导致不断震荡，甚至损失发散（Loss 爆炸）。

下面的代码使用 PyTorch 的自动求导模拟这三种情况。我们选定了 5 组具有代表性的学习率，将它们放置在一个 **3 行 × 5 列** 的图中实时更新：
- **第一行**：观察回归直线的动态拟合
- **第二行**：观察 MSE Loss 下降还是因为震荡而变大
- **第三行**：我们将使用强大的二维热力图！横轴为权重 $w$，纵轴为截距 $b$，颜色越亮（或背景的等高色阶）代表该处的理论 Loss 值越高，中心暗色则是最优解的“山谷”。留意白色轨迹，它精准记录了各个学习率在山下探索的脚印！

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output, display
import warnings
warnings.filterwarnings('ignore')

# ========================================================
# 1. 设定学习率和超参数
# ========================================================
lrs = [0.0001, 0.01, 0.03, 0.0485, 0.049]
labels = ['(极小)', '(合适)', '(最优)', '(震荡)', '(发散)']
lr_titles = [f'LR={lr} {labels[i]}' for i, lr in enumerate(lrs)]

w_start = -150.0
b_start = 150.0

epochs = 5000
plot_every = 100  # 每 100 轮刷新一次图表动画

# ========================================================
# 2. 提前计算 2D 热力图 (w, b) 
# ========================================================
w_min, w_max = -250, 250
b_min, b_max = -250, 250
param_num = 80

w_grid_1d = np.linspace(w_min, w_max, param_num)
b_grid_1d = np.linspace(b_min, b_max, param_num)
w_grid, b_grid = np.meshgrid(w_grid_1d, b_grid_1d)
loss_grid = np.zeros((param_num, param_num))

for i in range(param_num):
    for j in range(param_num):
        y_p = w_grid[i, j] * x + b_grid[i, j]
        loss_grid[i, j] = ((y_p - y)**2).mean() / 2

# 准备 PyTorch 张量
x_t = torch.tensor(x, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)
ws = [torch.tensor([w_start], requires_grad=True) for _ in range(5)]
bs = [torch.tensor([b_start], requires_grad=True) for _ in range(5)]

# ========================================================
# 3. 用于记录训练过程的容器
# ========================================================
hist_w = [[] for _ in range(5)]
hist_b = [[] for _ in range(5)]
hist_loss = [[] for _ in range(5)]
hist_epochs = []

x_line = np.array([x.min() - 0.5, x.max() + 0.5])
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

# ========================================================
# 4. 动态训练循环
# ========================================================
for epoch in range(epochs):
    for i in range(5):
        # 保护机制：如果因为学习率过大跑到无穷大，则停止更新它的图表
        if torch.isnan(ws[i]) or torch.isinf(ws[i]) or ws[i].abs() > 3000:
            hist_w[i].append(hist_w[i][-1] if len(hist_w[i])>0 else w_start)
            hist_b[i].append(hist_b[i][-1] if len(hist_b[i])>0 else b_start)
            hist_loss[i].append(hist_loss[i][-1] if len(hist_loss[i])>0 else 0)
            continue
            
        y_pred = ws[i] * x_t + bs[i]
        loss = ((y_pred - y_t)**2).mean() / 2
        loss.backward()
        
        with torch.no_grad():
            ws[i] -= lrs[i] * ws[i].grad
            bs[i] -= lrs[i] * bs[i].grad
            ws[i].grad.zero_()
            bs[i].grad.zero_()
            
        hist_w[i].append(ws[i].item())
        hist_b[i].append(bs[i].item())
        hist_loss[i].append(loss.item())
        
    hist_epochs.append(epoch + 1)
    
    # ================= 动态绘图 =================
    if (epoch + 1) % plot_every == 0 or epoch == epochs - 1:
        clear_output(wait=True)
        fig, axes = plt.subplots(3, 5, figsize=(22, 14))
        fig.suptitle(f"学习率对梯度下降轨迹的影响 (Epoch: {epoch+1}/{epochs})", fontsize=22, fontweight='bold')
        
        for col in range(5):
            w_current = hist_w[col][-1]
            b_current = hist_b[col][-1]
            loss_current = hist_loss[col][-1]
            
            # --- 第一行: 拟合结果 ---
            ax1 = axes[0][col]
            ax1.scatter(x, y, color=colors[0], alpha=0.5, edgecolors='k')
            clip_w = np.clip(w_current, -300, 300)
            clip_b = np.clip(b_current, -300, 300)
            ax1.plot(x_line, clip_w * x_line + clip_b, color='crimson', lw=2.5)
            ax1.set_title(lr_titles[col], fontsize=16)
            ax1.set_xlabel('Rooms')
            ax1.set_ylabel('Price')
            ax1.set_ylim([np.min(y)-5, np.max(y)+5])
            
            # --- 第二行: 损失变化情况 ---
            ax2 = axes[1][col]
            ax2.plot(hist_epochs, hist_loss[col], color=colors[1], lw=2)
            ax2.set_xlabel('Epoch')
            ax2.set_ylabel('Loss (MSE)')
            
            valid_losses = [l for l in hist_loss[col] if l > 0 and not np.isnan(l)]
            if len(valid_losses)>1 and (max(valid_losses)/(min(valid_losses)+1e-8)) > 100:
                ax2.set_yscale('log')
                
            # --- 第三行: W-B 2D 热力图上的变化情况 ---
            ax3 = axes[2][col]
            # 理论底图热力图 (使用 log 色阶让山谷更明显可以选用 vmax)
            im = ax3.imshow(loss_grid, extent=[w_min, w_max, b_min, b_max], origin='lower', cmap='viridis', zorder=0, aspect='auto', vmax=np.percentile(loss_grid, 95))
            
            # 实际移动路径
            ax3.plot(hist_w[col], hist_b[col], color='white', marker='.', markersize=4, lw=1.5, zorder=2)
            # 最新位置的高亮星星
            ax3.scatter([w_current], [b_current], color='red', s=200, marker='*', edgecolors='white', zorder=3)
            
            ax3.set_xlabel('Weight (w)')
            ax3.set_ylabel('Bias (b)')
            ax3.set_xlim([w_min, w_max])
            ax3.set_ylim([b_min, b_max])
            
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()

print("—"*50)
print("通过图表可以直观发现：")
print("1. 极小的学习率 (0.0001)：每一步移动极小，即使150轮在热力图上也只走了一小截。")
print("2. 合适与最优 (0.01-0.04)：稳稳朝着黄色的‘山谷中心’前进。")
print("3. 震荡学习率 (0.0485)：左右反复横跳，极易跑偏。")
print("4. 发散学习率 (0.049)：步子大，直接跳出热力图画布，Loss 火箭爆炸升空！")
print("—"*50)


### 🚀 进阶：为什么要使用自适应梯度？ (Adaptive Gradients)

在上一节中我们看到，**固定的学习率很难调节**：太小了跑不动，太大了容易震荡甚至发散 💥。此外，在拥有多个参数（例如 $w$ 和 $b$）的情况下，不同参数的最佳学习率往往也不一样。如果我们选用了一个比较大的学习率（如0.0489），在 $w$ 方向上可能会出现剧烈震荡，导致训练崩盘 🤯。

为了解决这个问题，我们可以引入 **自适应梯度算法（例如 Adagrad）** 🪄：
- **原理**：它会记录每个参数过去所有的梯度平方和（即历史陡峭程度）。
    $$G_{t} = G_{t-1} + g_t^2$$
- **更新规则**：将原学习率除以 $\sqrt{G_t + \epsilon}$。
    $$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{G_t + \epsilon}} g_t$$
- **优势**：对于梯度一直很大的参数，它的专属学习率会迅速**衰减**，防止震荡绕弯；对于梯度较小的参数，它的学习率则相对较大，加速收敛！🏃‍♂️💨

👇 下面，我们将同时使用 **基础梯度下降 (SGD)** 和 **自适应梯度 (Adagrad)**，比较在**优质学习率 (0.03)** 和 **震荡学习率 (0.0489)** 下的实际轨迹表现：\n

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

# ========================================================
# 1. 设定学习率和超参数
# ========================================================
lrs = [0.03, 0.03, 0.0489, 0.0489]
is_adagrad = [False, True, False, True]
titles = [
    'SGD (LR=0.03)\n稳步前进', 
    'Adagrad (LR=0.03)\n收敛更精准', 
    'SGD (LR=0.0489)\n严重震荡', 
    'Adagrad (LR=0.0489)\n化烂牌为神奇'
]

w_start = -150.0
b_start = 150.0

epochs = 5000
plot_every = 100  # 每 100 轮刷新一次图表动画

# ========================================================
# 2. 热力图底图准备
# ========================================================
w_min, w_max = -250, 250
b_min, b_max = -250, 250
param_num = 80

w_grid_1d = np.linspace(w_min, w_max, param_num)
b_grid_1d = np.linspace(b_min, b_max, param_num)
w_grid, b_grid = np.meshgrid(w_grid_1d, b_grid_1d)
loss_grid = np.zeros((param_num, param_num))

for i in range(param_num):
    for j in range(param_num):
        y_p = w_grid[i, j] * x + b_grid[i, j]
        loss_grid[i, j] = ((y_p - y)**2).mean() / 2

# 准备 PyTorch 张量
x_t = torch.tensor(x, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32)

ws = [torch.tensor([w_start], requires_grad=True) for _ in range(4)]
bs = [torch.tensor([b_start], requires_grad=True) for _ in range(4)]

# 在手动实现 Adagrad 时需要的历史梯度累计量 G
G_w = [torch.zeros(1) for _ in range(4)]
G_b = [torch.zeros(1) for _ in range(4)]
epsilon = 1e-8

# ========================================================
# 3. 记录日志的容器
# ========================================================
hist_w = [[] for _ in range(4)]
hist_b = [[] for _ in range(4)]
hist_loss = [[] for _ in range(4)]
hist_epochs = []

x_line = np.array([x.min() - 0.5, x.max() + 0.5])
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# ========================================================
# 4. 动态训练循环
# ========================================================
for epoch in range(epochs):
    for i in range(4):
        # 保护机制
        if torch.isnan(ws[i]) or torch.isinf(ws[i]) or ws[i].abs() > 3000:
            hist_w[i].append(hist_w[i][-1] if len(hist_w[i])>0 else w_start)
            hist_b[i].append(hist_b[i][-1] if len(hist_b[i])>0 else b_start)
            hist_loss[i].append(hist_loss[i][-1] if len(hist_loss[i])>0 else 0)
            continue
            
        y_pred = ws[i] * x_t + bs[i]
        loss = ((y_pred - y_t)**2).mean() / 2
        loss.backward()
        
        with torch.no_grad():
            if is_adagrad[i]:
                # Adagrad 的自适应下降逻辑
                G_w[i] += ws[i].grad ** 2
                G_b[i] += bs[i].grad ** 2
                ws[i] -= (lrs[i] / torch.sqrt(G_w[i] + epsilon)) * ws[i].grad
                bs[i] -= (lrs[i] / torch.sqrt(G_b[i] + epsilon)) * bs[i].grad
            else:
                # 基础版 SGD
                ws[i] -= lrs[i] * ws[i].grad
                bs[i] -= lrs[i] * bs[i].grad
                
            ws[i].grad.zero_()
            bs[i].grad.zero_()
            
        hist_w[i].append(ws[i].item())
        hist_b[i].append(bs[i].item())
        hist_loss[i].append(loss.item())
        
    hist_epochs.append(epoch + 1)
    
    # ================= 动态绘图 (3x4 Grid) =================
    if (epoch + 1) % plot_every == 0 or epoch == epochs - 1:
        clear_output(wait=True)
        fig, axes = plt.subplots(3, 4, figsize=(20, 14))
        fig.suptitle(f"基础 SGD vs 自适应梯度 Adagrad (Epoch: {epoch+1}/{epochs})", fontsize=22, fontweight='bold')
        
        for col in range(4):
            w_current = hist_w[col][-1]
            b_current = hist_b[col][-1]
            loss_current = hist_loss[col][-1]
            
            # --- 1. 拟合效果 ---
            ax1 = axes[0][col]
            ax1.scatter(x, y, color='teal', alpha=0.5, edgecolors='k')
            clip_w = np.clip(w_current, -300, 300)
            clip_b = np.clip(b_current, -300, 300)
            ax1.plot(x_line, clip_w * x_line + clip_b, color='crimson', lw=2.5)
            ax1.set_title(titles[col], fontsize=15)
            ax1.set_xlabel('Rooms')
            ax1.set_ylabel('Price')
            ax1.set_ylim([np.min(y)-5, np.max(y)+5])
            
            # --- 2. 损失曲线 ---
            ax2 = axes[1][col]
            ax2.plot(hist_epochs, hist_loss[col], color=colors[col], lw=2.5)
            ax2.set_xlabel('Epoch')
            ax2.set_ylabel('Loss (MSE)')
            if len([l for l in hist_loss[col] if l>0])>1 and (max(hist_loss[col])/(min([l for l in hist_loss[col] if l>0])+1e-8)) > 100:
                ax2.set_yscale('log')
                
            # --- 3. 热力图轨迹 ---
            ax3 = axes[2][col]
            im = ax3.imshow(loss_grid, extent=[w_min, w_max, b_min, b_max], origin='lower', cmap='viridis', aspect='auto', vmax=np.percentile(loss_grid, 95))
            ax3.plot(hist_w[col], hist_b[col], color='white', marker='.', markersize=4, lw=1.5)
            ax3.scatter([w_current], [b_current], color='red', s=200, marker='*', edgecolors='white', zorder=3)
            ax3.set_xlabel('Weight (w)')
            ax3.set_ylabel('Bias (b)')
            ax3.set_xlim([w_min, w_max])
            ax3.set_ylim([b_min, b_max])
            
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

print("—"*50)
print("🔑 核心观察：")
print("1. 当 LR=0.03 (最优学习率) 时：SGD 能走到谷底，但 Adagrad 自适应衰减步长，让路径更加平滑和精准。")
print("2. 当 LR=0.0489 (过大学习率) 时 🌟：如果在 SGD 下会引发剧烈震荡（红线飞出屏幕外）甚至失效；")
print("   👉 但是在 Adagrad 下，自适应机制在检测到震荡（巨大历史梯度累计）后立即衰减这方向的学习率，强行救回崩盘的训练进程！简直是化烂牌为神奇！")
print("—"*50)

### 🐢 为什么自适应梯度“失败”了？ (Adagrad 的真实面目)

刚才我们看到，无论是优质的 `0.03` 还是震荡的 `0.0489`，一旦套用上自适应梯度（Adagrad），模型仿佛瞬间被“冻结”了，参数几乎卡在原地不动！😭 难道 Adagrad 徒有虚名？

其实并不是！这正是由于 Adagrad 的核心机制带来的**学习率尺度变迁**：
- **在普通 SGD 中**：参数更新量 = 学习率 $\times$ 巨大的原始梯度（例如前几轮的梯度甚至高达几千）。因此 `0.03 \times 6000 = 180`，参数在一步之内可以移动数百个单位！
- **在 Adagrad 中**：梯度被自身历史累计的大小 $\sqrt{G_t}$ 归一化了！这导致 $\frac{g_t}{\sqrt{G_t}}$ 的值约等于 $1$ 或 $-1$。也就是说，**每一步的实际移动量最多就是指定的学习率 LR 本身**。
  
如果 LR=0.03，那么每一步参数最多只移动 0.03 个单位。而在我们的问题里，参数 $w$ 从初始的 `-200` 走到理想位置需要经过约 `200` 的长途跋涉，只给 0.03 的步伐显然是“龟速挪动”。

👉 **正确开启的秘密：提高基础学习率**
使用自适应梯度算法时，学习率的物理意义更接近于“每步允许的最大移动步长”。如果我们给 Adagrad 提供一个充足的“车费”（例如 `LR = 100` 或 `LR = 50`），它就能凭借自适应衰减机制发挥出真正的威力！\n

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

# 比较 Adagrad (LR=0.03) vs Adagrad (LR=100)
lrs = [0.03, 100.0]
is_adagrad = [True, True]
titles = [
    'Adagrad (LR=0.03)\n受到历史梯度限制，寸步难行', 
    'Adagrad (LR=100.0)\n获得充足基础步长，满血复活！'
]

w_start = -150.0
b_start = 150.0

epochs = 5000
plot_every = 100  # 每 100 轮刷新一次图表动画

ws = [torch.tensor([w_start], requires_grad=True) for _ in range(2)]
bs = [torch.tensor([b_start], requires_grad=True) for _ in range(2)]
G_w = [torch.zeros(1) for _ in range(2)]
G_b = [torch.zeros(1) for _ in range(2)]
epsilon = 1e-8

hist_w, hist_b, hist_loss = [[] for _ in range(2)], [[] for _ in range(2)], [[] for _ in range(2)]
hist_epochs = []
colors = ['#1f77b4', '#d62728']

for epoch in range(epochs):
    for i in range(2):
        if torch.isnan(ws[i]) or torch.isinf(ws[i]) or ws[i].abs() > 3000:
            hist_w[i].append(hist_w[i][-1] if len(hist_w[i])>0 else w_start)
            hist_b[i].append(hist_b[i][-1] if len(hist_b[i])>0 else b_start)
            hist_loss[i].append(hist_loss[i][-1] if len(hist_loss[i])>0 else 0)
            continue
            
        y_pred = ws[i] * x_t + bs[i]
        loss = ((y_pred - y_t)**2).mean() / 2
        loss.backward()
        
        with torch.no_grad():
            if is_adagrad[i]:
                G_w[i] += ws[i].grad ** 2
                G_b[i] += bs[i].grad ** 2
                ws[i] -= (lrs[i] / torch.sqrt(G_w[i] + epsilon)) * ws[i].grad
                bs[i] -= (lrs[i] / torch.sqrt(G_b[i] + epsilon)) * bs[i].grad
            else:
                ws[i] -= lrs[i] * ws[i].grad
                bs[i] -= lrs[i] * bs[i].grad
                
            ws[i].grad.zero_()
            bs[i].grad.zero_()
            
        hist_w[i].append(ws[i].item())
        hist_b[i].append(bs[i].item())
        hist_loss[i].append(loss.item())
        
    hist_epochs.append(epoch + 1)
    
    if (epoch + 1) % plot_every == 0 or epoch == epochs - 1:
        clear_output(wait=True)
        fig, axes = plt.subplots(3, 2, figsize=(14, 14))
        fig.suptitle(f"找到真理：合理的自适应梯度对比 (Epoch: {epoch+1}/{epochs})", fontsize=20, fontweight='bold')
        
        for col in range(2):
            w_current = hist_w[col][-1]
            b_current = hist_b[col][-1]
            loss_current = hist_loss[col][-1]
            
            # --- 1. 拟合效果 ---
            ax1 = axes[0][col]
            ax1.scatter(x, y, color='teal', alpha=0.5, edgecolors='k')
            clip_w = np.clip(w_current, -300, 300)
            clip_b = np.clip(b_current, -300, 300)
            ax1.plot(x_line, clip_w * x_line + clip_b, color='crimson', lw=2.5)
            ax1.set_title(titles[col], fontsize=15)
            ax1.set_xlabel('Rooms')
            ax1.set_ylabel('Price')
            
            # --- 2. 损失曲线 ---
            ax2 = axes[1][col]
            ax2.plot(hist_epochs, hist_loss[col], color=colors[col], lw=2.5)
            ax2.set_xlabel('Epoch')
            ax2.set_ylabel('Loss (MSE)')
            if len([l for l in hist_loss[col] if l>0])>1 and (max(hist_loss[col])/(min([l for l in hist_loss[col] if l>0])+1e-8)) > 100:
                ax2.set_yscale('log')
                
            # --- 3. 热力图轨迹 ---
            ax3 = axes[2][col]
            im = ax3.imshow(loss_grid, extent=[w_min, w_max, b_min, b_max], origin='lower', cmap='viridis', aspect='auto', vmax=np.percentile(loss_grid, 95))
            ax3.plot(hist_w[col], hist_b[col], color='white', marker='.', markersize=4, lw=1.5)
            ax3.scatter([w_current], [b_current], color='red', s=200, marker='*', edgecolors='white', zorder=3)
            ax3.set_xlabel('Weight (w)')
            ax3.set_ylabel('Bias (b)')
            ax3.set_xlim([w_min, w_max])
            ax3.set_ylim([b_min, b_max])
            
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

### 🚀 更快更稳的下坡哲学：动量法 (Momentum)

除了自适应梯度之外，还有一种非常直观且经典的优化策略——**动量法 (Momentum)**。

**🤔 为什么需要动量？**
回忆一下在山谷中滚落的物理现象。如果参数只是每次按照“当前位置的坡度”走一步（犹如普通的梯度下降），遇到陡峭的横向峡谷时，就会在两侧**剧烈震荡**；而在平坦的纵向谷底时，又会**进展缓慢**。
此时，如果我们引入**物理学中的“惯性/动量”**概念：让参数更新保留上一时刻的“速度方向”，就能很好地解决这个问题！
- 横向的震荡由于方向正负交替，会被**相互抵消**；
- 纵向的下坡由于方向始终一致，会被**不断加速**！

![image.png](images/gradient_01.png)

**🔬 更新规则**：
1. 计算当前速度 $v_{t} = \gamma v_{t-1} + \eta g_t$ （$\gamma$ 为动量衰减系数，通常设为 0.9）。
2. 更新参数 $\theta_{t+1} = \theta_t - v_t$。

👇 **对比大揭秘**：我们将使用同一个会立刻引起震荡的偏大学习率 (`0.0489`)，对比 **基础 SGD** 和 **带 Momentum 的 SGD ($\gamma=0.9$)** 的走势。您会看到，动量不仅抚平了横向震荡，还凭借累积的速度更快地冲向了终点！🏎️💨\n

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

# 我们统一使用容易导致震荡的偏大学习率 (LR=0.0489)
lr = 0.0489
gamma = 0.9  # 动量的衰减系数

lrs = [lr, lr]
use_momentum = [False, True]
titles = [
    f'SGD (LR={lr})\n没有动量：反复横跳剧烈震荡', 
    f'SGD + Momentum (LR={lr})\n动量抵消震荡，加速冲刺！'
]

w_start = -150.0
b_start = 150.0

epochs = 5000
plot_every = 100  # 每 100 轮刷新一次图表动画

ws = [torch.tensor([w_start], requires_grad=True) for _ in range(2)]
bs = [torch.tensor([b_start], requires_grad=True) for _ in range(2)]

# 初始化动量速度变量 V
v_w = [torch.zeros(1) for _ in range(2)]
v_b = [torch.zeros(1) for _ in range(2)]

hist_w, hist_b, hist_loss = [[] for _ in range(2)], [[] for _ in range(2)], [[] for _ in range(2)]
hist_epochs = []
colors = ['#d62728', '#2ca02c']

for epoch in range(epochs):
    for i in range(2):
        if torch.isnan(ws[i]) or torch.isinf(ws[i]) or ws[i].abs() > 3000:
            hist_w[i].append(hist_w[i][-1] if len(hist_w[i])>0 else w_start)
            hist_b[i].append(hist_b[i][-1] if len(hist_b[i])>0 else b_start)
            hist_loss[i].append(hist_loss[i][-1] if len(hist_loss[i])>0 else 0)
            continue
            
        y_pred = ws[i] * x_t + bs[i]
        loss = ((y_pred - y_t)**2).mean() / 2
        loss.backward()
        
        with torch.no_grad():
            if use_momentum[i]:
                # 加入动量的更新逻辑： V_t = gamma * V_{t-1} + LR * g_t
                v_w[i] = gamma * v_w[i] + lrs[i] * ws[i].grad
                v_b[i] = gamma * v_b[i] + lrs[i] * bs[i].grad
                ws[i] -= v_w[i]
                bs[i] -= v_b[i]
            else:
                # 基础版 SGD
                ws[i] -= lrs[i] * ws[i].grad
                bs[i] -= lrs[i] * bs[i].grad
                
            ws[i].grad.zero_()
            bs[i].grad.zero_()
            
        hist_w[i].append(ws[i].item())
        hist_b[i].append(bs[i].item())
        hist_loss[i].append(loss.item())
        
    hist_epochs.append(epoch + 1)
    
    if (epoch + 1) % plot_every == 0 or epoch == epochs - 1:
        clear_output(wait=True)
        fig, axes = plt.subplots(3, 2, figsize=(14, 14))
        fig.suptitle(f"基础震荡 SGD vs 动量加速 Momentum (Epoch: {epoch+1}/{epochs})", fontsize=20, fontweight='bold')
        
        for col in range(2):
            w_current = hist_w[col][-1]
            b_current = hist_b[col][-1]
            loss_current = hist_loss[col][-1]
            
            # --- 1. 拟合效果 ---
            ax1 = axes[0][col]
            ax1.scatter(x, y, color='teal', alpha=0.5, edgecolors='k')
            clip_w = np.clip(w_current, -300, 300)
            clip_b = np.clip(b_current, -300, 300)
            ax1.plot(x_line, clip_w * x_line + clip_b, color='crimson', lw=2.5)
            ax1.set_title(titles[col], fontsize=15)
            ax1.set_xlabel('Rooms')
            ax1.set_ylabel('Price')
            
            # --- 2. 损失曲线 ---
            ax2 = axes[1][col]
            ax2.plot(hist_epochs, hist_loss[col], color=colors[col], lw=2.5)
            ax2.set_xlabel('Epoch')
            ax2.set_ylabel('Loss (MSE)')
            if len([l for l in hist_loss[col] if l>0])>1 and (max(hist_loss[col])/(min([l for l in hist_loss[col] if l>0])+1e-8)) > 100:
                ax2.set_yscale('log')
                
            # --- 3. 热力图轨迹 ---
            ax3 = axes[2][col]
            im = ax3.imshow(loss_grid, extent=[w_min, w_max, b_min, b_max], origin='lower', cmap='viridis', aspect='auto', vmax=np.percentile(loss_grid, 95))
            ax3.plot(hist_w[col], hist_b[col], color='white', marker='.', markersize=4, lw=1.5)
            ax3.scatter([w_current], [b_current], color='yellow', s=200, marker='*', edgecolors='k', zorder=3)
            ax3.set_xlabel('Weight (w)')
            ax3.set_ylabel('Bias (b)')
            ax3.set_xlim([w_min, w_max])
            ax3.set_ylim([b_min, b_max])
            
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

## 📚 模型泛化能力 (Generalization) 与过拟合灾难

**🤔 为什么需要泛化？**
当我们训练一个模型时，我们在**已知**的数据（训练集）上不断调整参数以降低误差。但这本身有极大的风险：就像一个考前“死记硬背”了所有练习题答案的学生，虽然练习题全对，但在真正大考（**未知**的数据、测试集）时却一塌糊涂！
**泛化能力 (Generalization)** 就是衡量模型在“从未见过的新数据”上预测准确度的能力。

👇 **过拟合灾难模拟：极少样本 + 高阶复杂模型**
为了动态展现泛化失败的典型场景（**过拟合 Overfitting**），我们制造一个**灾难性的组合**：
1. **极少的数据**：只取仅仅 10 个数据点作为训练集。
2. **极复杂的模型**：最高采用 9 阶多项式曲线 ($x^9$)。

*(注：为了让 9 阶多项式在梯度下降中能够顺畅收敛，我们对多项式的所有高阶项进行了 **标准差归一化 (Standard Scaler)**，并使用了我们刚刚学过的 **Adagrad+Momentum (Adam)** 来全速狂飙！)*
让我们来欣赏模型为了迎合这 10 个红点，是如何一步步变得极度畸形的！💥

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

# 1. 抽取单维特征 (Rooms 平均房间数) 以实现 2D 平面可视化拟合
x1 = data[:, 5].astype(np.float32)

# 2. 为了模型训练的绝对收敛稳定性，我们对输入的特征 x 和偏置 y 都需要彻底归一化
# 特别是因为后续我们要生成多项式，如果不先归一化，x^9 瞬间就会产生数值爆炸！
x1_norm = (x1 - np.mean(x1)) / np.std(x1)
y_norm = (y - np.mean(y)) / np.std(y)

# 3. 把所有的真实数据切分为训练集(80%) 和 测试集(20%)
X_tr_all, X_te_all, y_tr_all, y_te_all = train_test_split(x1_norm.reshape(-1, 1), y_norm, test_size=0.2, random_state=42)

# ================= 过拟合灾难级设定 =================
# 我们极度吝啬地【仅仅】从高质量训练集中抽出 10 个蓝点！
np.random.seed(42)
idx_tr = np.random.choice(len(X_tr_all), 10, replace=False)
X_tr_small, y_tr_small = X_tr_all[idx_tr], y_tr_all[idx_tr]

# 同时我们随机抽取 50 个未曾谋面的绿点作为无情的测试考官
idx_te = np.random.choice(len(X_te_all), 50, replace=False)
X_te_small, y_te_small = X_te_all[idx_te], y_te_all[idx_te]

# 用于画曲线（连贯红线）的紧密底层 X
x_line_np = np.linspace(min(x1_norm)-0.5, max(x1_norm)+0.5, 300).reshape(-1, 1)

# 我们要对比的三种模型阶数：1阶（呆板），2阶（优雅），9阶（变态走火入魔模式）
degrees = [1, 2, 9]

X_trs_t, X_tes_t, X_lines_t, ws, bs, optimizers = [], [], [], [], [], []

# ================= 为每一个模型做特征矩阵的预处理与 Adam 优化器安排 =================
for d in degrees:
    # 建立 sklearn 模型获取 1~d 阶的新多项特征
    poly = PolynomialFeatures(degree=d, include_bias=False)
    scaler = StandardScaler()
    
    # 🌟 核心技巧：对哪怕是已经规范化的 x 生成的 x^2...x^9，再次执行全面列向量标准化 (Standard Scaler)！
    # 避免在梯度下降时，因为 x^9 的跳跃级数值造成梯度要么消失要么爆炸！这至关重要！
    P_tr = scaler.fit_transform(poly.fit_transform(X_tr_small))
    P_te = scaler.transform(poly.transform(X_te_small))
    P_line = scaler.transform(poly.transform(x_line_np))
    
    X_trs_t.append(torch.tensor(P_tr, dtype=torch.float32))
    X_tes_t.append(torch.tensor(P_te, dtype=torch.float32))
    X_lines_t.append(torch.tensor(P_line, dtype=torch.float32))
    
    # 构建当前模型的“参数小车”(由于带有需要不断调整更新的值，注意开启 requires_grad_)
    w_init = (torch.randn((d, 1)) * 0.1).clone().detach().requires_grad_(True)
    b_init = torch.zeros(1, requires_grad=True)
    ws.append(w_init)
    bs.append(b_init)
    
    # 🌟 核心技巧：全面拥抱大名鼎鼎的终极智能引擎 —— Adam (Adaptive Moment Estimation) 优化器
    # 之前我们痛苦于：偏小的 lr 走不动，过大的 lr 会剧烈震荡跳出山谷。
    # Adam 将“动量法”(保持方向惯性，防左右横跳) 与 “Adagrad”(惩罚震荡量，加速慢量) 包裹于一身！
    # 它在训练如此荒诞的 9 阶多项式时，能如推土机一般势如破竹！
    optimizer = torch.optim.Adam([w_init, b_init], lr=0.1)
    optimizers.append(optimizer)

y_tr_t = torch.tensor(y_tr_small, dtype=torch.float32)
y_te_t = torch.tensor(y_te_small, dtype=torch.float32)

epochs = 10000
plot_every = 100 # 每跑 100 步停下来画一下状态，形成流畅的动态小动画

# ================= 全速开动动态训练循环 =================
for epoch in range(epochs):
    tr_errs = []
    te_errs = []
    
    for i in range(3):
        # 1. 模型的前向猜测 Y_predict (用张量矩阵乘法高效计算出 10 个测试点各自的推测位置)
        Y_pred = torch.mm(X_trs_t[i], ws[i]).squeeze() + bs[i]
        
        # 2. 平均二分之一次方差 (MSE 损失评价)
        loss = ((Y_pred - y_tr_t)**2).mean() / 2
        
        # 3. PyTorch 反向传播三部曲 (清除旧数据 -> 求出所有关联节点导数 -> 让 Adam 根据混合经验主动踩一脚油门)
        optimizers[i].zero_grad()
        loss.backward()
        optimizers[i].step() # Adam 根据自身包含的 Momentum 和 Adagrad 完美推算出的步长自动更新了权重。
            
        with torch.no_grad():
            tr_errs.append(loss.item() * 2) 
            y_pred_te = torch.mm(X_tes_t[i], ws[i]).squeeze() + bs[i]
            te_errs.append(((y_pred_te - y_te_t)**2).mean().item())
            
    # ================= 动态绘图：揭示扭曲灾难的形成 =================
    if (epoch + 1) % plot_every == 0 or epoch == epochs - 1:
        clear_output(wait=True)
        # 注意：每次都在新回合内部生成并立即销毁图表句柄，彻底避免后台残留幽灵空白框 <Figure size ...>！
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        fig.suptitle(f"模型复杂度 vs 过拟合灾难 动态训练 (全场由 Adam 引擎接管)  Epoch: {epoch+1}/{epochs}", fontsize=20, fontweight='bold')
        
        titles = [f"p=1 (欠拟合)\n一条笔直的死脑筋", f"p=2 (较为合适)\n平滑流畅的黄金抛物线", f"p=9 (恐怖过拟合!)\n为了连点而陷入妖娆弯折"]
        for col in range(3):
            with torch.no_grad():
                curr_line_pred = (torch.mm(X_lines_t[col], ws[col]).squeeze() + bs[col]).numpy()
            
            # --- 【上半区】：模型面对可怜兮兮训练集的狂热谄媚表现 ---
            ax1 = axes[0][col]
            ax1.scatter(X_tr_small, y_tr_small, color='dodgerblue', s=80, edgecolors='k', label='可怜可悲的 10 个蓝色真实点')
            ax1.plot(x_line_np, curr_line_pred, color='crimson', lw=2.5)
            ax1.set_title(f"{titles[col]}\nTrain MSE: {tr_errs[col]:.4f}", fontsize=15)
            ax1.set_ylim(-3, 4)
            ax1.legend(loc='lower right')
            ax1.grid(True, alpha=0.3)
            
            # --- 【下半区】：模型过度膨胀扭曲后，盲猜完全脱离绿点轨道的凄惨现状 ---
            ax2 = axes[1][col]
            ax2.scatter(X_te_small, y_te_small, color='forestgreen', s=40, edgecolors='k', alpha=0.9, label='海量无情测试考官点')
            ax2.plot(x_line_np, curr_line_pred, color='crimson', lw=2.5, label='拟合曲线')
            ax2.set_ylim(-3, 4)
            ax2.set_xlabel('Rooms (Normalized)')
            ax2.set_ylabel('Price')
            ax2.set_title(f"Test MSE: {te_errs[col]:.1f}", fontsize=15)
            ax2.legend(loc='lower right')
            ax2.grid(True, alpha=0.3)
            
        plt.tight_layout(rect=[0, 0, 1, 0.95])
        plt.show()

print("—"*50)
print("🔑 动态变形的究极扭曲 (Adam + DataScaler 的功劳)：")
print("1. 在极其强劲的 Adam 下：那条 p=9 的红线甚至不需要跑一两万圈，仅用两三千圈，就像一条贪吃蛇一样以极其妖娆扭曲的身姿死死包住了这可悲的 10 个蓝色真实点！Train MSE 近乎为 0！")
print("2. 这正是最可怕的：当这种扭曲发疯的折线去大胆猜绿色的大面区域时，分数不仅是不及格，简直是全盘崩溃 (Test MSE 飞出了天际)！📉")
print("—"*50)

## 🛡️ 拯救过拟合大作战

刚才我们在极其匮乏的 10 个数据点上使用高阶 9阶多项式，领教了令人毛骨悚然的“超级折线灾难”。接下来，让我们看看如何把走火入魔的模型治愈并拉回正轨！

### 🗡️ 方法一：使用更简单的模型 (奥卡姆剃刀)
**奥卡姆剃刀原理**认为："如无必要，勿增实体"。在上一节的中心图里我们已经发现，仅使用 p=2（简单的抛物线）就能在仅有 10 个样本的数据集上取得极其出色的测试成绩。
**结论**：不要盲目追求高阶数、高参数量的模型。**模型复杂度必须与数据多样性严格匹配。**

### 📈 方法二：增加真实样本数量 (海量数据镇压)
如果现实中可以获得更多的真实数据，即使是高达 9 阶甚至更疯癫的方程，也能被海量的数据“结结实实地死死吸附在主流趋势上”，避免它在空缺区间乱抽风！
👇 我们释放出全部 400 个蓝色的真实训练样本，进行 6000 轮动态训练，看看原本疯癫的 p=9 曲线是如何被大军无情压制的！

In [ ]:
# 1. 直接铺开原本全部的 400 余个可用蓝色真实特征资源
poly9_full = PolynomialFeatures(degree=9, include_bias=False)
scaler_full = StandardScaler()

# 2. 数据升维并紧接着由 StandardScaler 进行标准降魔压制
P_tr_full = scaler_full.fit_transform(poly9_full.fit_transform(X_tr_all))
P_line_full = scaler_full.transform(poly9_full.transform(x_line_np))

X_tr_full_t = torch.tensor(P_tr_full, dtype=torch.float32)
y_tr_full_t = torch.tensor(y_tr_all, dtype=torch.float32)
X_line_full_t = torch.tensor(P_line_full, dtype=torch.float32)

# 3. 在全新的模型世界打造全新的车架子 (初始化张量节点)
w_f = (torch.randn((9, 1)) * 0.1).clone().detach().requires_grad_(True)
b_f = torch.zeros(1, requires_grad=True)

# 统一使用被证实战果拔群的 Adam 智能引擎调校参数
optimizer_f = torch.optim.Adam([w_f, b_f], lr=0.1)

# 开展全量数据下的大决战：让 p=9 这匹猛兽看看 400 个点密密麻麻的锁链阵压制力
epochs_f = 10000
for epoch in range(epochs_f):
    # 简单的打分求偏导步骤
    Y_pred = torch.mm(X_tr_full_t, w_f).squeeze() + b_f
    loss = ((Y_pred - y_tr_full_t)**2).mean() / 2
    
    optimizer_f.zero_grad()
    loss.backward()
    optimizer_f.step()
        
    # 为了防范残留 <Figure size> 幽灵，依然把 fig 控制在判定块内新建展示！
    if (epoch + 1) % 100 == 0 or epoch == epochs_f - 1:
        clear_output(wait=True)
        fig_f, ax_f = plt.subplots(figsize=(10, 6))
        
        with torch.no_grad():
            line_pred_full = (torch.mm(X_line_full_t, w_f).squeeze() + b_f).numpy()
            
        ax_f.scatter(X_tr_all, y_tr_all, color='dodgerblue', alpha=0.4, label='海量的镇压巨石：400 多个真实特征点！')
        ax_f.plot(x_line_np, line_pred_full, color='crimson', lw=3, label='曾不可一世的 p=9 曲线，被活生生压平！')
        
        tr_err_f = loss.item() * 2
        ax_f.set_title(f"海量真实样本大军压境：动态训练镇压纪实 Epoch {epoch+1}\nTrain MSE: {tr_err_f:.3f}", fontsize=16, fontweight='bold')
        ax_f.set_ylim(-3, 4)
        ax_f.grid(True, alpha=0.3)
        ax_f.legend()
        plt.show()

print("🎯 看到了吗！连曾经不可一世、飞天遁地扭曲的 p=9 死亡曲线，在绝对的海量数据锚点压制下，也被剥夺了折弯翻滚的空间，只能乖乖贴地变成一条优美平滑的顺服走势！")

### 🧬 方法三：数据增强 (生成虚拟数据云团)
如果实在没有那么多钱去买真实数据呢？我们可以借用 **数据增强 (Data Augmentation)**。就像在图片识别时把一张原图稍加旋转一样，在数值型数据里，我们可以在仅有的 10 个孤立散点周围，注入一点点高斯随机噪声，强行捏出 100 个橙色的“虚拟星云点”！
👇 让我们看看 100 个生成的星云虚拟点，能不能拉拽住发疯的 9阶曲线！

In [ ]:
# 如果只有 10 个源点，不如利用现实采样扰动的灵感，给它们注入一点点偏差虚拟出无穷的影子点！
aug_x1, aug_y = [], []

# 我们用两重循环实现一化百的魔法裂变！
for i in range(10):
    for _ in range(100):
        # 加上微小随机波动的高斯噪音
        aug_x1.append(X_tr_small[i,0] + np.random.normal(0, 0.3))
        aug_y.append(y_tr_small[i] + np.random.normal(0, 0.3))

# 合成虚拟军团
X_aug = np.array(aug_x1).reshape(-1, 1)
y_aug = np.array(aug_y)

poly9_aug = PolynomialFeatures(degree=9, include_bias=False)
scaler_aug = StandardScaler()

P_tr_aug = scaler_aug.fit_transform(poly9_aug.fit_transform(X_aug))
P_line_aug = scaler_aug.transform(poly9_aug.transform(x_line_np))

X_aug_t = torch.tensor(P_tr_aug, dtype=torch.float32)
y_aug_t = torch.tensor(y_aug, dtype=torch.float32)
X_line_aug_t = torch.tensor(P_line_aug, dtype=torch.float32)

w_a = (torch.randn((9, 1)) * 0.1).clone().detach().requires_grad_(True)
b_a = torch.zeros(1, requires_grad=True)

# 召唤负责快速调参的 Adam !
optimizer_a = torch.optim.Adam([w_a, b_a], lr=0.1)

epochs_a = 10000
for epoch in range(epochs_a):
    Y_pred = torch.mm(X_aug_t, w_a).squeeze() + b_a
    loss = ((Y_pred - y_aug_t)**2).mean() / 2
    
    optimizer_a.zero_grad()
    loss.backward()
    optimizer_a.step()
        
    if (epoch + 1) % 500 == 0 or epoch == epochs_a - 1:
        clear_output(wait=True)
        fig_a, ax_a = plt.subplots(figsize=(10, 6))
        
        with torch.no_grad():
            line_pred_a = (torch.mm(X_line_aug_t, w_a).squeeze() + b_a).numpy()
            
        ax_a.scatter(X_tr_small, y_tr_small, color='k', s=250, marker='*', label='可怜孤独的起源：原始 10 个真实核心黑星')
        ax_a.scatter(X_aug, y_aug, color='orange', s=35, alpha=0.7, edgecolors='none', label='使用奥义裂变出的庞大橙色虚拟星云点')
        ax_a.plot(x_line_np, line_pred_a, color='crimson', lw=3, label='原本应该四处逃窜的 p=9 死亡红线被星云牢牢拽死！')
        
        ax_a.set_title(f"数据增强 (Data Augmentation)：用虚空生星云来阻扼曲线扭曲！ (Epoch {epoch+1})", fontsize=16, fontweight='bold')
        ax_a.set_ylim(-3, 4)
        ax_a.grid(True, alpha=0.3)
        ax_a.legend()
        plt.show()

print("🌟 非常奇妙对不对！仅仅是在那 10 颗孤星周围捏出了一百颗散沙般的噪音点（橙色），但这 100 颗沙子却拥有海一样巨大的摩擦力和宽广的视野。即使是 p=9 这匹野马也不敢轻易横跳跨出星云外围，它的猖狂扭曲行为遭到了史诗级别的削弱！")

### 💎 万一发生欠拟合了，可以尝试引入更丰富的特征
单看房屋的“平均房间数”，信息量太有限了。如果只有少数样本，引入更多特征反而容易导致过拟合。但是，当我们不仅拥有单一维度特征，甚至还在前文中通过数据增强大幅扩充了数据量后，单变量模型很可能会因为容量不足而在这茫茫数据海中出现**欠拟合**。

与其试图用单变量强扭高维曲线，不如我们直接引入**波士顿房价原本的 13 维全特征**（犯罪率、税率等）来提高模型的解答能力！
用多维线性模型 $y = w_1x_1 + w_2x_2 \dots + w_{13}x_{13} + b$，既不增加多项式阶数，又极大地充实了判定依据。这就好比一个人不仅看长相，还看性格、学识等多维画像！

>(我们在原有的全量全特征矩阵上执行简单线性回归)


In [ ]:
x_all_norm = (data - np.mean(data, axis=0)) / np.std(data, axis=0) # 13 维全特征！
y_all_norm = y_norm

x_mul_tr, x_mul_te, y_mul_tr, y_mul_te = train_test_split(x_all_norm, y_all_norm, test_size=0.2, random_state=42)
x_mul_t = torch.tensor(x_mul_tr, dtype=torch.float32)
y_mul_t = torch.tensor(y_mul_tr, dtype=torch.float32)
x_mul_te_t = torch.tensor(x_mul_te, dtype=torch.float32)
y_mul_te_t = torch.tensor(y_mul_te, dtype=torch.float32)

# ==========================================
# 1. 训练全特征模型 (13个维度)
# ==========================================
w_mul = (torch.randn((13, 1)) * 0.1).clone().detach().requires_grad_(True)
b_mul = torch.zeros(1, requires_grad=True)

# 统一使用 Adam 优化器
optimizer_mul = torch.optim.Adam([w_mul, b_mul], lr=0.1)

for epoch in range(5000):
    y_pred = torch.mm(x_mul_t, w_mul).squeeze() + b_mul
    loss = ((y_pred - y_mul_t)**2).mean() / 2
    
    optimizer_mul.zero_grad()
    loss.backward()
    optimizer_mul.step()

with torch.no_grad():
    y_pred_te = torch.mm(x_mul_te_t, w_mul).squeeze() + b_mul
    test_loss_mul = ((y_pred_te - y_mul_te_t)**2).mean() / 2

# ==========================================
# 2. 训练单特征模型 (仅用[5]平均房间数)
# ==========================================
# 取出第 5 列，保持形状为 (n, 1)
x_sin_t = x_mul_t[:, 5].unsqueeze(1)
x_sin_te_t = x_mul_te_t[:, 5].unsqueeze(1)

w_sin = (torch.randn((1, 1)) * 0.1).clone().detach().requires_grad_(True)
b_sin = torch.zeros(1, requires_grad=True)

optimizer_sin = torch.optim.Adam([w_sin, b_sin], lr=0.1)

for epoch in range(5000):
    y_pred_sin = torch.mm(x_sin_t, w_sin).squeeze() + b_sin
    loss_sin = ((y_pred_sin - y_mul_t)**2).mean() / 2
    
    optimizer_sin.zero_grad()
    loss_sin.backward()
    optimizer_sin.step()

with torch.no_grad():
    y_pred_sin_te = torch.mm(x_sin_te_t, w_sin).squeeze() + b_sin
    test_loss_sin = ((y_pred_sin_te - y_mul_te_t)**2).mean() / 2

print("【模型效果对比】")
print(f"单特征回归 (仅用“平均房间数”) - 测试集误差 (MSE): {test_loss_sin.item():.4f}")
print(f"全特征回归 (13维全量特征)     - 测试集误差 (MSE): {test_loss_mul.item():.4f}")
print("=> 结论：引入更丰富的特征，直接提供了判断房价的更多依据，从而大幅降低了误差，提升了预测准确度！")


### ⚖️ 可否继续优化一下？引入正则化 (Regularization) 对特征影响力进行调优！
当我们不知道哪些特征是有效特征，哪些是无用特征时，模型很容易为了迎合极少数样本而去“背诵”那些毫无意义的特征（比如房屋门牌号尾数），这就是一种隐蔽的过拟合。
为了模拟这一点，我们在 13 维全特征上**加入 2 个完全是由 numpy.random 生成的“毫无意义的噪声列”**。
此时，我们采用 **L1/L2 正则化**，它会在目标函数上加一个惩罚项（惩罚 $w$ 变得过大）。模型必须对最有用的特征分配有限的权重。
*结果是：它能够自动识别出这 2 列是废物特征，并将其权重降至零周边！*

In [ ]:
# =====================================================================
# 引入噪声特征以观察正则化效果
# =====================================================================
# 向 13 维原始真实特征追加 2 列纯随机噪声（服从标准正态分布），凑齐 15 维
noise_cols = np.random.normal(0, 1, size=(x_all_norm.shape[0], 2))
x_noise_aug = np.hstack([x_all_norm, noise_cols])

# 重新切分含有噪声特征的训练集和测试集
x_n_tr, x_n_te, y_n_tr, y_n_te = train_test_split(x_noise_aug, y_all_norm, test_size=0.2, random_state=42)

# 转换为 PyTorch 张量
x_n_t = torch.tensor(x_n_tr, dtype=torch.float32)
y_n_t = torch.tensor(y_n_tr, dtype=torch.float32)
x_n_te_t = torch.tensor(x_n_te, dtype=torch.float32)
y_n_te_t = torch.tensor(y_n_te, dtype=torch.float32)

def train_regularized_model(reg_type="L1", lambda_reg=0.05, epochs=5000, lr=0.1):
    '''
    一个用于训练带正则化线性回归的辅助函数。
    reg_type: 'L1' (Lasso) 或 'L2' (Ridge)
    '''
    # 随机初始化由 15 个输入的权重 (13个真实特征 + 2个噪声)，以及 1 个偏置项
    w = (torch.randn((15, 1)) * 0.1).clone().detach().requires_grad_(True)
    b = torch.zeros(1, requires_grad=True)
    
    # 采用 Adam 优化器
    optimizer = torch.optim.Adam([w, b], lr=lr)
    
    for epoch in range(epochs):
        # 1. 前向传播：计算预测值
        y_pred = torch.mm(x_n_t, w).squeeze() + b
        
        # 2. 计算基础 MSE 训练误差
        mse_loss = ((y_pred - y_n_t)**2).mean() / 2
        
        # 3. 施加正则化惩罚项
        if reg_type == "L1":
            # L1 正则化 (Lasso) = lambda * sum(|w|)
            # 偏好使得非核心权值直接收敛到 0，自带“特征选择”功能
            loss = mse_loss + lambda_reg * w.abs().sum()
        elif reg_type == "L2":
            # L2 正则化 (Ridge) = lambda * sum(w^2)
            # 偏好让所有权重整体变小、更平滑，但很难彻底收敛到 0
            loss = mse_loss + lambda_reg * (w**2).sum()
        
        # 4. 反向传播与优化
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    # 计算此时模型在测试集的数据表现（泛化能力）
    with torch.no_grad():
        # 基于已训练的 w, b 对测试集进行预测
        y_pred_te = torch.mm(x_n_te_t, w).squeeze() + b
        # 测试集只看实际差距，不加惩罚项
        test_mse = ((y_pred_te - y_n_te_t)**2).mean() / 2
        # 也算一下训练集最终的不带正则项的基础MSE
        final_train_mse = ((torch.mm(x_n_t, w).squeeze() + b - y_n_t)**2).mean() / 2
        
    # 返回分离出计算图后的纯权重数值，以及这两种 Loss
    return w.detach().squeeze().numpy(), final_train_mse.item(), test_mse.item()

# =====================================================================
# 分别训练 L1 和 L2 正则化模型并提取泛化表现
# =====================================================================
# 惩罚系数 (lambda) 皆设定为 0.05 进行比较
w_l1, train_loss_l1, test_loss_l1 = train_regularized_model("L1", 0.05)
w_l2, train_loss_l2, test_loss_l2 = train_regularized_model("L2", 0.05)

print("【L1 正则化 (Lasso)】")
print(f"训练集基础误差: {train_loss_l1:.4f}  |  测试集泛化误差: {test_loss_l1:.4f}")
print("【L2 正则化 (Ridge)】")
print(f"训练集基础误差: {train_loss_l2:.4f}  |  测试集泛化误差: {test_loss_l2:.4f}\n")

# =====================================================================
# 柱状图可视化：观察 L1 和 L2 对权重的不同打压机制
# =====================================================================
features = fts_names + ['无用噪声 1', '无用噪声 2']

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# ---- 图 1：L1 正则化条形图 ----
bars_l1 = axes[0].bar(range(15), np.abs(w_l1), color='teal', alpha=0.7)
# 突出显示最后2根噪声条
bars_l1[-2].set_color('crimson')
bars_l1[-1].set_color('crimson')
axes[0].set_xticks(range(15))
axes[0].set_xticklabels(features, rotation=45, ha='right', fontsize=12)
axes[0].set_ylabel("权重的绝对值大小 |W|", fontsize=14)
axes[0].set_title("L1 正则化：自动特征选择 (多数无效特征被强压至0)", fontsize=16)
axes[0].grid(axis='y', alpha=0.3)

# ---- 图 2：L2 正则化条形图 ----
bars_l2 = axes[1].bar(range(15), np.abs(w_l2), color='royalblue', alpha=0.7)
bars_l2[-2].set_color('crimson')
bars_l2[-1].set_color('crimson')
axes[1].set_xticks(range(15))
axes[1].set_xticklabels(features, rotation=45, ha='right', fontsize=12)
axes[1].set_ylabel("权重的绝对值大小 |W|", fontsize=14)
axes[1].set_title("L2 正则化：收缩权重 (噪声权重虽受打压但依然存在)", fontsize=16)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


### L1 与 L2 正则化的结果与优缺点对比

![image.png](images/regularization_01.png)

从上方的拟合与权重表现来看，当我们为模型添加这两种不同类型的惩罚（正则项）时，它们起到了迥然不同的作用。

* **L1 正则化 (Lasso) 的惩罚机制**：惩罚权重的绝对值之和，即 $\lambda \sum |w_i|$
* **L2 正则化 (Ridge) 的惩罚机制**：惩罚权重的平方和，即 $\lambda \sum w_i^2$

结合上述公式，我们可以通过下表总结理解两者在实际表现中的差异：

| 特性对比 | **L1 正则化 (Lasso)** | **L2 正则化 (Ridge)** |
|:---|:---|:---|
| **对权重的影响** | 导致稀疏权重，许多无效或次要特征的权重会**直接变为 0** | 导致连续且平滑缩小，绝大多数权重会**变小**但不会归 0 |
| **主要优点** | 自带**特征选择**功能，特别适合高维数据且剔除无用的噪声维度，模型更为精简具有可解释性 | 在特征高度共线性时极其稳定，通常能给出整体**表现最好、泛化能力强**的鲁棒模型 |
| **主要缺点** | 会过滤掉部分有关联的特征（可能丢失轻微有效的信息） | 无法做特征选择，最后模型需要所有输入参与计算，计算开销大 |
| **建议适用场景**| 特征维度极高、存在大量无关/噪声特征、需要找出关键特征时 | 特征维度相对不多，或者认为大部分特征对预测都有用、需要防止模型对某个特征过度依赖时 |


> **一句话总结**：L1 像是一位雷厉风行的裁剪师，专门淘汰混日子的无用特征（如我们加入的随机噪声被直接全部压到 0）；L2 则像一个和稀泥的调停者，不允许任何单独特征一家独大，强迫所有关联特征“平摊”权重和责任。


### 🛑 早停法 (Early Stopping)
既然过拟合是因为对训练数据“学得太深、钻了牛角尖”，那我们只要趁它还没“走火入魔”时提早喊停就好了！
这就是**早停法**：我们在训练途中，不仅看训练集的误差（一直在降），同时并行监控**验证集**的误差。如果在某个 Epoch 点，验证集误差降到最低并且**开始反弹飙升**，那我们立刻停止训练！

In [ ]:
# =====================================================================
# 完全摒弃多项式，只用纯粹的多个特征展示早停法 (Early Stopping)
# 特征选择：8 维关键特征 (包含房间数、师生比等)
# =====================================================================
# 从全量标准化数据中提取这 8 个特征
selected_features_idx = [0, 4, 5, 7, 8, 10, 11, 12]
x_8_norm = x_all_norm[:, selected_features_idx]

# 切分：训练集，验证集，测试集
x_tr_8, x_temp_8, y_tr_8, y_temp_8 = train_test_split(x_8_norm, y_all_norm, test_size=0.4, random_state=42)
# 把剩下的 0.4 再对半分为验证集(0.2)和测试集(0.2)
x_val_8, x_te_8, y_val_8, y_te_8 = train_test_split(x_temp_8, y_temp_8, test_size=0.5, random_state=42)

# 不使用多项式组合！直接用纯纯的线性回归！
# 提取 50 个样本，在 8 维特征下，过少的样本依然会导致模型把训练集的特有噪声学进去
x_tr_pt = torch.tensor(x_tr_8[:50], dtype=torch.float32)
y_tr_pt = torch.tensor(y_tr_8[:50], dtype=torch.float32)
x_val_pt = torch.tensor(x_val_8, dtype=torch.float32)
y_val_pt = torch.tensor(y_val_8, dtype=torch.float32)
x_te_pt = torch.tensor(x_te_8, dtype=torch.float32)
y_te_pt = torch.tensor(y_te_8, dtype=torch.float32)

num_features = x_tr_pt.shape[1]

# =====================================================================
# 保证与模型 1 同宗同源：提前生成绝对一模一样的初始起点权重
# =====================================================================
w_init = (torch.randn((num_features, 1)) * 0.1).clone().detach()
b_init = torch.zeros(1)

# =====================================================================
# 模型 1：正常跑满 3000 Epoch (无早停)
# =====================================================================
w_full = w_init.clone().detach().requires_grad_(True)
b_full = b_init.clone().detach().requires_grad_(True)
opt_full = torch.optim.Adam([w_full, b_full], lr=0.01)

tr_hist_full, val_hist_full = [], []
for epoch in range(3000):
    y_pred = torch.mm(x_tr_pt, w_full).squeeze() + b_full
    loss = ((y_pred - y_tr_pt)**2).mean() / 2
    
    with torch.no_grad():
        val_pred = torch.mm(x_val_pt, w_full).squeeze() + b_full
        val_loss = ((val_pred - y_val_pt)**2).mean() / 2
        tr_hist_full.append(loss.item())
        val_hist_full.append(val_loss.item())
        
    opt_full.zero_grad()
    loss.backward()
    opt_full.step()

with torch.no_grad():
    test_pred_full = torch.mm(x_te_pt, w_full).squeeze() + b_full
    test_loss_full = ((test_pred_full - y_te_pt)**2).mean() / 2

# =====================================================================
# 模型 2：带有早停法 (Early Stopping)
# =====================================================================
# 取用未经任何训练的初始随机权重
w_es = w_init.clone().detach().requires_grad_(True)
b_es = b_init.clone().detach().requires_grad_(True)
opt_es = torch.optim.Adam([w_es, b_es], lr=0.01)

best_val_loss = float('inf')
patience = 100 
patience_counter = 0
best_epoch = 0
best_w, best_b = None, None

for epoch in range(3000):
    y_pred = torch.mm(x_tr_pt, w_es).squeeze() + b_es
    loss = ((y_pred - y_tr_pt)**2).mean() / 2
    
    with torch.no_grad():
        val_pred = torch.mm(x_val_pt, w_es).squeeze() + b_es
        val_loss = ((val_pred - y_val_pt)**2).mean() / 2
        
        # 判断是否刷新最优验证集误差
        if val_loss.item() < best_val_loss:
            best_val_loss = val_loss.item()
            best_epoch = epoch
            patience_counter = 0
            best_w, best_b = w_es.clone(), b_es.clone()
        else:
            patience_counter += 1
            
    if patience_counter >= patience:
        break
        
    opt_es.zero_grad()
    loss.backward()
    opt_es.step()

with torch.no_grad():
    # 用保存下来的历史最佳 w 和 b 迎接测试集大考
    test_pred_es = torch.mm(x_te_pt, best_w).squeeze() + best_b
    test_loss_es = ((test_pred_es - y_te_pt)**2).mean() / 2

# =====================================================================
# 打印效果并绘制对比图表
# =====================================================================
import matplotlib.pyplot as plt
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
plt.rcParams['axes.unicode_minus'] = False # 彻底解决负号 Glyph 找不到的 Warning

print("【测试集盲猜效果超级对比】")
print(f"不采用早停 (硬跑3000轮) 慢慢过拟合 MSE = {test_loss_full.item():.4f}")
print(f"采用早停法 (跑{best_epoch}轮退出) 保住防线 MSE = {test_loss_es.item():.4f}")
print(f"-> 完全不依靠多项式升维！仅靠 8 个特征在线性叠加下，早停法依然通过停止记忆特有噪声，成功拿到了更低的测试集误差。")

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True, sharey=True)

# 限制坐标系确保 U 型视觉底端更明显
axes[0].set_ylim(0, max(val_hist_full[0] * 1.5, 0.4)) 

axes[0].plot(tr_hist_full, label='Train Loss (越降越低)', color='dodgerblue')
axes[0].plot(val_hist_full, label='Val Loss (U型飙升)', color='darkorange')
axes[0].set_title("无早停：模型强行记忆训练集，后期出现过拟合")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(tr_hist_full[:best_epoch+patience], label='Train Loss', color='dodgerblue')
axes[1].plot(val_hist_full[:best_epoch+patience], label='Val Loss', color='darkorange')
axes[1].axvline(best_epoch, color='crimson', linestyle='--', label=f'早停点 (Epoch {best_epoch})')
axes[1].set_title("带有早停：防微杜渐，精准切在验证集误差的最低谷")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 🔄 终极泛化测试仪：交叉验证 (Cross-Validation)

在前面的验证过程中，我们一直使用固定的 `train_test_split` 把数据分为训练集和验证集（或测试集）。这会带来一个隐患：**如果某次随机划分碰巧把容易预测的数据全分进了测试集，模型表现就会在这个固定划分上“虚高”**！

特别是当数据量很少的时候，不同的分割方案可能会让评估结果差异巨大。这时候，我们需要 **K-折交叉验证 (K-Fold Cross-Validation)** 闪亮登场 ✨：
- **核心思想**：不采用固定的一刀切，而是把数据集平均分成 K 份。让这 K 份数据轮流担当“主考官（测试集）”，其余 K-1 份联合起来作为训练集。
- **好处**：保证每个样本都有过一次做测试集的机会！最终取 K 次预测误差的**平均值**，就能得到模型真实且极其稳定的实力评估！📊

👇 **下面我们使用最常用的 5-Fold 交叉验证，测试我们基础的线性回归模型在完整的 13 维波士顿特征数据上的真实泛化实力。**\n

In [ ]:
import torch
import numpy as np
from sklearn.model_selection import KFold
import warnings
warnings.filterwarnings('ignore')

# 我们对之前的 13维全特征矩阵 x_all_norm 进行 5折交叉验证
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

fold_results = []
fold_no = 1

print("="*60)
print(f"【评估方法一】 启动 {k_folds}-Fold (K折) 交叉验证...")
print("="*60)

for train_idx, test_idx in kf.split(x_all_norm):
    
    x_tr_fold = torch.tensor(x_all_norm[train_idx], dtype=torch.float32)
    y_tr_fold = torch.tensor(y_norm[train_idx], dtype=torch.float32)
    x_te_fold = torch.tensor(x_all_norm[test_idx], dtype=torch.float32)
    y_te_fold = torch.tensor(y_norm[test_idx], dtype=torch.float32)
    
    # 每次折叠必须初始化全新的权重
    w_fold = (torch.randn((13, 1)) * 0.1).clone().detach().requires_grad_(True)
    b_fold = torch.zeros(1, requires_grad=True)
    
    optimizer_fold = torch.optim.Adam([w_fold, b_fold], lr=0.1)
    
    for epoch in range(3000):
        y_pred = torch.mm(x_tr_fold, w_fold).squeeze() + b_fold
        loss = ((y_pred - y_tr_fold)**2).mean() / 2
        
        optimizer_fold.zero_grad()
        loss.backward()
        optimizer_fold.step()
            
    with torch.no_grad():
        y_pred_te = torch.mm(x_te_fold, w_fold).squeeze() + b_fold
        loss_te = ((y_pred_te - y_te_fold)**2).mean() / 2
        fold_results.append(loss_te.item())
        
    print(f" 第 {fold_no} 折 - 测试误差 (MSE): {loss_te.item():.4f}")
    fold_no += 1


print("\n" + "="*60)
print(f"【评估方法二】 启动 Bootstrapping (自助法) 重采样验证...")
print("="*60)

n_samples = len(x_all_norm)
n_bootstraps = 5
boot_results = []

# 固定随机种子保证结果可复现
np.random.seed(42)

for i in range(n_bootstraps):
    # 随机有放回抽样形成训练集
    train_idx = np.random.choice(n_samples, n_samples, replace=True)
    # 取出未被抽中的样本（袋外数据 OOB, Out-Of-Bag）作为天然的验证集
    test_idx = np.array(list(set(range(n_samples)) - set(train_idx)))
    
    x_tr_boot = torch.tensor(x_all_norm[train_idx], dtype=torch.float32)
    y_tr_boot = torch.tensor(y_norm[train_idx], dtype=torch.float32)
    x_te_boot = torch.tensor(x_all_norm[test_idx], dtype=torch.float32)
    y_te_boot = torch.tensor(y_norm[test_idx], dtype=torch.float32)
    
    # 每次 Bootstrap 必须初始化全新的权重
    w_boot = (torch.randn((13, 1)) * 0.1).clone().detach().requires_grad_(True)
    b_boot = torch.zeros(1, requires_grad=True)
    
    optimizer_boot = torch.optim.Adam([w_boot, b_boot], lr=0.1)
    
    for epoch in range(3000):
        y_pred = torch.mm(x_tr_boot, w_boot).squeeze() + b_boot
        loss = ((y_pred - y_tr_boot)**2).mean() / 2
        
        optimizer_boot.zero_grad()
        loss.backward()
        optimizer_boot.step()
            
    with torch.no_grad():
        y_pred_te = torch.mm(x_te_boot, w_boot).squeeze() + b_boot
        loss_te = ((y_pred_te - y_te_boot)**2).mean() / 2
        boot_results.append(loss_te.item())
        
    print(f" 第 {i+1} 次 Bootstrap (验证集未抽中样本量:{len(test_idx)}) - 测试误差: {loss_te.item():.4f}")

print("\n" + "—"*60)
print(f" 交叉验证与 Bootstrapping 的终极能力评估对比：")
print(f" [K-Fold] 交叉验证 ({k_folds} 折) 的平均 MSE 误差为：{np.mean(fold_results):.4f}")
print(f" [Bootstrapping] 自助法 ({n_bootstraps} 次) 的平均 OOB MSE 误差为：{np.mean(boot_results):.4f}")
print("—"*60)


### K-Fold (K折交叉验证) 与 Bootstrapping (自助法) 的核心对比

这两种验证方法通常被用作高级的机器学习评测标杆（不让单个数据切分的运气影响模型评判）。它们的核心目的都是**在有限的数据集中，最大化地发掘数据反复组合后的评估价值**。具体区别可以参考下表：

| 方法对比 | **K-Fold 交叉验证** | **Bootstrapping 自助法验证** |
|:---|:---|:---|
| **采样方式** | **无放回**抽样。将数据集均分为等量 $K$ 份，用 $K-1$ 份训练，剩余 1 份验证，循环直至覆盖全集 | **有放回**抽样。每次从 $N$ 个样本中随机抽取 $N$ 次作为训练集，大约永远有约 $1/e \approx 36.8\%$ 的样本始终未被抽中（即袋外数据 OOB， Out-Of-Bag）作为本次的验证集 |
| **数据利用效率** | 所有的原始样本都会被**严丝合缝地用作测试集恰好一次**，绝不重叠 | 随机性强：样本可能在一轮中被多次重复抽中训练，也可能根本未被抽中 |
| **方差与偏差 (Bias-Variance)** | 极其严谨，倾向于给出近乎无偏的模型能力估计值，极大地降低评价的系统性偏差 (Bias低) | 虽然引入了一些打乱重组的评估偏差，但由于数据集在多次采样中重合度大，方差往往很小（即多次重跑平均后非常稳健一致） |
| **计算复杂度边界** | 计算开销量被完全锁死在固定为 $K$ 次完整的模型训练上 | 极其灵活：计算开销由人为设定的循环采样次数决定，爱测多少次测多少次 |
| **主要优点** | 公平、严谨、不重不漏，是目前大多数顶尖论文及真实工业界最严苛的标准验证法 | 第一：对**极小微型数据集**非常宽容，因为它用放回抽样强行“无中生有”变出了大量不同的平行宇宙版本的数据；第二：天生自带随机扰动基因，是随机森林和所有 Bagging 模型集成学习的底层基石 |
| **主要缺点** | 面对极端微型数据集时（例如只有 50 条命），留出的那仅仅 1 份验证集可能只有个位数，很容易导致每一折指标由于基数太小剧烈震荡 | “有放回”抽取的设定打乱了数据原生的分布比例或者少数类占比，它的 OOB袋外测试数据并不能从数学上 100% 严谨地反映出外界大自然的真实分布规律 |
